In [115]:
# Install Essential Libraries:
# !pip install google-generativeai scikit-learn ipywidgets requests beautifulsoup4 numpy

In [116]:
import google.generativeai as genai
import os
import requests
from bs4 import BeautifulSoup
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from typing import List, Dict, Tuple, Optional
import json
import hashlib
import warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 1. API Configuration:

# Google AI Studio API key
API_KEY = "omitted"

try:
    genai.configure(api_key=API_KEY)
    print("API Key configured successfully.")
except Exception as e:
    print(f"Error configuring API Key: {e}")
    raise ValueError("API key configuration failed.")

API Key configured successfully.


In [118]:
# Define embedding and generative models
EMBEDDING_MODEL = "models/text-embedding-004"
GENERATION_MODEL = "models/gemini-1.5-flash-latest"

In [119]:
# Global variables
documents_data = []
query_history = []
agent_choices = []
correct_choices = []

In [120]:
# 2. Data Loading Function (for Web Content):
def web_fetcher(url: str) -> Optional[str]:
    try:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        # BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        for script in soup(["script", "style"]):
            script.decompose()
        
        text = soup.get_text()
        
        lines = (line.strip() for line in text.splitlines())
        chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
        text = ' '.join(chunk for chunk in chunks if chunk)
        
        return text
        
    except requests.exceptions.ConnectionError:
        print(f"     Connection error for {url}")
        return None
    except requests.exceptions.HTTPError:
        print(f"     HTTP error for {url}")
        return None
    except requests.exceptions.Timeout:
        print(f"     Timeout error for {url}")
        return None
    except Exception as e:
        print(f"     Error for {url}: {str(e)}")
        return None

In [121]:
# 3.1 Chunking Function:
def chunky(text: str, chunk_size: int = 800, overlap: int = 100) -> List[str]:
    if not text:
        return []
    
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        if end >= len(text): # append remaining text
            chunks.append(text[start:])
            break
        
        # Break at periods and whitespaces
        chunk = text[start:end]
        last_period = chunk.rfind('.')
        last_space = chunk.rfind(' ')
        
        if last_period > len(chunk) * 0.8:
            end = start + last_period + 1
        elif last_space > len(chunk) * 0.8:
            end = start + last_space
        
        chunks.append(text[start:end])
        start = end - overlap
        
        # Prevent infinite loop
        if start <= 0:
            start = end
    
    return chunks

In [122]:
# 3.2 Embedding Generation
def embedder(text: str) -> np.ndarray:
    if not text:
        return np.array([])
    try:
        # sync
        response = genai.embed_content(
            model=EMBEDDING_MODEL,
            content=text,
            # task_type="retrieval_document" // Good for improving quality but supports only text-embedding-005, text-multilingual-embedding-002, gemini-embedding-001
        )
        return np.array(response['embedding'])
    except Exception as e:
        print(f"Error embedding {e}")
        print(f"Check API Key or model '{EMBEDDING_MODEL}'")
        return np.array([]) # Empty

In [123]:
# 3.3 Similarity Calculation
def cos_similarity(embedding1: np.ndarray, embedding2: np.ndarray) -> float:
    if embedding1.size == 0 or embedding2.size == 0:
        return 0.0
    
    try:
        # sklearn.metrics.pairwise.cosine_similarity
        embedding1_reshaped = embedding1.reshape(1, -1)
        embedding2_reshaped = embedding2.reshape(1, -1)
        similarity = cosine_similarity(embedding1_reshaped, embedding2_reshaped)[0][0]
        return float(similarity)
    except Exception as e:
        print(f"Error calculating similarity: {e}")
        return 0.0

In [124]:
# 4. Data Storage - Bonus: Caching Embeddings
cache_path = "embeddings_cache.json"

# Load the cache if it exists
if os.path.exists(cache_path):
    with open(cache_path, "r") as f:
        embedding_cache = json.load(f)
else:
    embedding_cache = {}

In [125]:
def get_chunk_key(text):
    return hashlib.sha256(text.encode()).hexdigest()

In [126]:
def store_web(urls: List[str], output_widget) -> None:
    global documents_data
    documents_data = []
    
    output_widget.clear_output()
    with output_widget:
        print("Processing web sources...")

    for i, url in enumerate(urls, 1):
        with output_widget:
            print(f"\nProcessing source {i}/{len(urls)}: {url}")
        
        # Full extracted text
        text = web_fetcher(url)
        if not text:
            with output_widget:
                print(f"Failed to load content from {url}")
            continue
        
        # Chunks
        chunks = chunky(text)
        with output_widget:
            print(f"Split into {len(chunks)} chunks")
        
        # Chunk embeddings (with caching)
        with output_widget:
            print("Generating chunk embeddings...")
        
        chunk_embeddings = []
        for j, chunk in enumerate(chunks):
            key = get_chunk_key(chunk)
            is_cached = key in embedding_cache
            if is_cached:
                embedding = np.array(embedding_cache[key]["embedding"])
            else:
                embedding = embedder(chunk)
                embedding_cache[key] = {
                    "text": chunk,
                    "embedding": embedding.tolist() if hasattr(embedding, "tolist") else embedding
                }
            if embedding.size > 0:
                chunk_embeddings.append(embedding)
            if (j + 1) % 5 == 0 or (j + 1) == len(chunks):
                msg = (
                    f"Cached embedding for chunk {j+1}/{len(chunks)}"
                    if is_cached else
                    f"Generated embedding for chunk {j+1}/{len(chunks)}"
                )
                with output_widget:
                    print(msg)

        # Save updated cache
        with open(cache_path, "w") as f:
            json.dump(embedding_cache, f)
        
        # Summary embedding
        summary_text = text[:1000] + "..." if len(text) > 1000 else text
        summary_key = get_chunk_key(summary_text)
        if summary_key in embedding_cache:
            summary_embedding = np.array(embedding_cache[summary_key]["embedding"])
        else:
            summary_embedding = embedder(summary_text)
            embedding_cache[summary_key] = {
                "text": summary_text,
                "embedding": summary_embedding.tolist() if hasattr(summary_embedding, "tolist") else summary_embedding
            }
            with open(cache_path, "w") as f:
                json.dump(embedding_cache, f)
        
        doc_data = {
            'url': url,
            'source_name': f"Source_{i}",
            'text': text,
            'chunks': chunks,
            'chunk_embeddings': chunk_embeddings,
            'summary_embedding': summary_embedding,
            'summary_text': summary_text
        }
        documents_data.append(doc_data)
        
        with output_widget:
            print(f"Source {i} processed successfully!")
    
    with output_widget:
        print(f"\nProcessed {len(documents_data)} web sources successfully!")

In [127]:
# 5. Agent Decision Module (Routing Logic):
def sel_source(query: str, output_widget) -> Tuple[str, float, int]:
    if not documents_data:
        return "", 0.0, -1
    
    with output_widget:
        print("\nAGENT DECISION PROCESS")
        print("=" * 40)
        print("Agent is evaluating each source...")
    
    # Query embedding
    query_embedding = embedder(query)
    if query_embedding.size == 0:
        with output_widget:
            print("Could not generate query embedding")
        return "", 0.0, -1
    
    best_source = ""
    best_similarity = 0.0
    best_index = -1
    
    # Iterate through all loaded web sources
    for i, doc_data in enumerate(documents_data):
        source_name = doc_data['source_name']
        
        # Compare query embedding with summary embedding
        similarity = cos_similarity(query_embedding, doc_data['summary_embedding'])
        
        with output_widget:
            print(f"Evaluating {source_name}...")
            print(f"{source_name}: {similarity:.2%} relevance")
        
        # Select source
        if similarity > best_similarity:
            best_similarity = similarity
            best_source = source_name
            best_index = i
    
    # Crucially, add a visual indication
    with output_widget:
        print(f"\nAGENT DECISION: Chose {best_source}")
        print(f"Source URL: {documents_data[best_index]['url']}")
    
    return best_source, best_similarity, best_index

In [128]:
# 6. Retrieval Module
def retrieve_chunks(query: str, source_index: int, top_k: int = 3) -> List[str]:
    if source_index < 0 or source_index >= len(documents_data):
        return []
    
    doc_data = documents_data[source_index]
    query_embedding = embedder(query)
    
    if query_embedding.size == 0:
        return doc_data['chunks'][:top_k]  # Return first 3 chunks as fallback
    
    chunk_similarities = []
    for i, chunk_embedding in enumerate(doc_data['chunk_embeddings']):
        similarity = cos_similarity(query_embedding, chunk_embedding)
        chunk_similarities.append((i, similarity))
    
    # Sort by similarity and get top_k (3)
    chunk_similarities.sort(key=lambda x: x[1], reverse=True)
    top_chunks = [doc_data['chunks'][i] for i, _ in chunk_similarities[:top_k]]
    
    return top_chunks

In [129]:
# 7. LLM Integration (Generative Response):
def generate_response(query: str, retrieved_chunks: List[str], source_info: Dict) -> str:
    if not retrieved_chunks:
        return "No relevant chunks retrieved for the question."
    
    context = "\n\n".join(retrieved_chunks)
    prompt = (
        f"You are a knowledgeable AI assistant. Carefully answer the user's question using ONLY the information provided in the context below. "
        f"If the answer cannot be found in the context, clearly state that the information is not available.\n"

        f"Context from {source_info['source_name']} ({source_info['url']}): {context}\n"

        f"User's Question: {query}\n"
        f"Your Answer:"
    )
    
    try:
        model = genai.GenerativeModel(GENERATION_MODEL)
        response = model.generate_content(prompt)
        
        answer = response.text
        reference = f"\n\nAnswer derived from: {source_info['source_name']}"
        reference += f"\nSource URL: {source_info['url']}"
        reference += f"\nRelevance: {source_info.get('relevance', 'N/A')}"
        
        return answer # + reference
        
    except Exception as e:
        return f"Error generating content: {e}"

In [130]:
# 8. Agent Performance Evaluation
def calc_metrics() -> Dict[str, float]:
    if not agent_choices or not correct_choices:
        return {}
    
    # Accuracy
    correct_predictions = sum(1 for agent, correct in zip(agent_choices, correct_choices) if agent == correct)
    accuracy = correct_predictions / len(agent_choices)
    
    all_sources = list(set(agent_choices + correct_choices))
    
    source_metrics = {}
    for source in all_sources:
        # True positives: agent chose this source and it was correct
        tp = sum(1 for agent, correct in zip(agent_choices, correct_choices) 
                if agent == source and correct == source)
        
        # False positives: agent chose this source but it was wrong
        fp = sum(1 for agent, correct in zip(agent_choices, correct_choices) 
                if agent == source and correct != source)
        
        # False negatives: agent didn't choose this source but should have
        fn = sum(1 for agent, correct in zip(agent_choices, correct_choices) 
                if agent != source and correct == source)
        
        # Metrics
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        source_metrics[source] = {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn
        }
    
    # Averages
    macro_precision = np.mean([metrics['precision'] for metrics in source_metrics.values()])
    macro_recall = np.mean([metrics['recall'] for metrics in source_metrics.values()])
    macro_f1 = np.mean([metrics['f1'] for metrics in source_metrics.values()])
    
    return {
        'accuracy': accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'source_metrics': source_metrics
    }

In [131]:
# 9. Interactive User Interface (using ipywidgets)
class UserInterface:
    def __init__(self):
        self.setup_widgets()
        self.setup_callbacks()
    
    def setup_widgets(self):
        # input fields for specifying multiple web URLs (e.g., comma-separated).
        self.url_input = widgets.Textarea(
            value='https://en.wikipedia.org/wiki/Space_exploration,https://en.wikipedia.org/wiki/Climate_change,https://en.wikipedia.org/wiki/Artificial_intelligence',
            placeholder='Enter comma-separated URLs (at least 3)',
            description='Web URLs:',
            layout=widgets.Layout(width='99%', height='80px')
        )
        
        # Add a "Load Sources" button
        self.load_button = widgets.Button(
            description='Load Sources',
            button_style='primary',
            layout=widgets.Layout(width='150px')
        )
        
        # Include a "Query:" input box
        self.query_input = widgets.Text(
            placeholder='Enter your question here',
            description='Query:',
            layout=widgets.Layout(width='99%')
        )
        
        # Submit Query
        self.query_button = widgets.Button(
            description='Submit Query',
            button_style='success',
            layout=widgets.Layout(width='150px')
        )
        
        # Feedback
        self.feedback_dropdown = widgets.Dropdown(
            options=[],
            description='Correct Source:',
            disabled=True,
            layout=widgets.Layout(width='300px')
        )

        self.feedback_button = widgets.Button(
            description='Submit Correction',
            button_style='warning',
            disabled=True,
            layout=widgets.Layout(width='150px')
        )

        self.agent_was_correct = widgets.ToggleButton(
            value=False,
            description='Agent was correct',
            disabled=True,
            button_style='warning',
            layout=widgets.Layout(width='150px')
        )
        
        # Output
        self.progress_output = widgets.Output(
            layout=widgets.Layout(
                height='200px',
                width='100%',
                overflow='auto',
                overflow_wrap='break-word',
                white_space='pre-wrap'
            )
        )
        self.response_output = widgets.Output(
            layout=widgets.Layout(
                height='400px',
                width='100%',
                overflow='auto',
                overflow_wrap='break-word',
                white_space='pre-wrap'
            )
        )
        self.evaluation_output = widgets.Output(
            layout=widgets.Layout(
                height='400px',
                width='100%',
                overflow='auto',
                overflow_wrap='break-word',
                white_space='pre-wrap'
            )
        )
        # Evaluation
        self.eval_button = widgets.Button(
            description='Show Evaluation',
            button_style='info',
            layout=widgets.Layout(width='150px')
        )
        
        self.current_agent_choice = None
        self.current_query = None
        
    def setup_callbacks(self):
        self.load_button.on_click(self.load_sources)
        self.query_button.on_click(self.submit_query)
        self.query_input.on_submit(self.submit_query) # Enter key = submit
        self.feedback_button.on_click(self.submit_feedback)
        self.agent_was_correct.observe(self.on_agent_correct_toggle, names='value')
        self.eval_button.on_click(self.show_evaluation)

        
    def load_sources(self, button):
        urls_text = self.url_input.value.strip()
        if not urls_text:
            with self.progress_output:
                print("Please enter URLs")
            return
        
        urls = [url.strip() for url in urls_text.split(',') if url.strip()]
        
        if len(urls) < 3:
            with self.progress_output:
                print("Please provide at least 3 URLs")
            return
        
        # Disable controls during loading
        self.load_button.disabled = True
        self.query_button.disabled = True
        
        try:
            store_web(urls, self.progress_output)
            
            # Enable query controls
            self.query_button.disabled = False
            
            # Update sources dropdown with sources
            if documents_data:
                source_options = [(doc['source_name'], doc['source_name']) for doc in documents_data]
                self.feedback_dropdown.options = source_options
                
        except Exception as e:
            with self.progress_output:
                print(f"Error loading sources: {e}")
        finally:
            self.load_button.disabled = False
            
    def submit_query(self, button_or_text_widget):
        query = self.query_input.value.strip()
        if not query:
            with self.response_output:
                print("Please enter a query.")
            return
        
        if not documents_data:
            with self.response_output:
                print("Please load sources first.")
            return
        
        # Clear previous outputs
        self.response_output.clear_output()
        
        # Store query for evaluation
        query_history.append(query)
        self.current_query = query
        
        # Agent selects source
        agent_choice, relevance, source_index = sel_source(query, self.response_output)
        
        if source_index < 0:
            with self.response_output:
                print("Agent could not select a suitable source.")
            return
        
        # Store agent choice
        agent_choices.append(agent_choice)
        self.current_agent_choice = agent_choice
        
        # Retrieve relevant chunks
        retrieved_chunks = retrieve_chunks(query, source_index)
        
        # Generate response
        source_info = documents_data[source_index].copy()
        source_info['relevance'] = f"{relevance:.2%}"
        
        response = generate_response(query, retrieved_chunks, source_info)
        
        # Display response
        with self.response_output:
            print("-" * 40)
            print("RESPONSE:")
            display(HTML(f"{response}"))
            print("-" * 40)
        
        # Enable feedback controls
        self.feedback_dropdown.disabled = False
        self.feedback_button.disabled = False
        self.agent_was_correct.disabled = False
        self.agent_was_correct.value = False
        
        # Set dropdown to agent's choice
        self.feedback_dropdown.value = agent_choice
        
    def on_agent_correct_toggle(self, change):
        if change['new']:  # Agent was correct
            self.feedback_dropdown.value = self.current_agent_choice
            self.feedback_dropdown.disabled = True
            # Auto submit feedback if agent was correct
            self.submit_feedback(None)
        else:  # Agent was wrong
            self.feedback_dropdown.disabled = False
            
    def submit_feedback(self, button):
        if self.current_agent_choice is None:
            with self.response_output:
                print("No query to provide feedback for.")
            return
        
        correct_choice = self.feedback_dropdown.value
        correct_choices.append(correct_choice)
        
        with self.response_output:
            if self.current_agent_choice == correct_choice:
                print("Feedback: Agent chose correctly.")
            else:
                print(f"Feedback: Agent chose {self.current_agent_choice}, but {correct_choice} was correct.")
        
        # Disable feedback controls
        self.feedback_dropdown.disabled = True
        self.feedback_button.disabled = True
        self.agent_was_correct.disabled = True
        
        # Clear query input
        self.query_input.value = ""
        
    def show_evaluation(self, button):
        self.evaluation_output.clear_output()
        
        if not correct_choices:
            with self.evaluation_output:
                print("No feedback data available for evaluation.")
            return
        
        metrics = calc_metrics()
        
        with self.evaluation_output:
            print("AGENT PERFORMANCE EVALUATION")
            print("=" * 50)
            print(f"Overall Metrics (Averages):")
            print(f"   Accuracy: {metrics['accuracy']:.2%}")
            print(f"   Macro Precision: {metrics['macro_precision']:.2%}")
            print(f"   Macro Recall: {metrics['macro_recall']:.2%}")
            print(f"   Macro F1-Score: {metrics['macro_f1']:.2%}")
            
            print(f"\nPer-Source Performance:")
            for source, source_metrics in metrics['source_metrics'].items():
                print(f"\n   {source}:")
                print(f"      Precision: {source_metrics['precision']:.2%}")
                print(f"      Recall: {source_metrics['recall']:.2%}")
                print(f"      F1-Score: {source_metrics['f1']:.2%}")
                print(f"      TP: {source_metrics['tp']}, FP: {source_metrics['fp']}, FN: {source_metrics['fn']}")
            
            print(f"\nTotal Queries Evaluated: {len(correct_choices)}")
            
            print(f"\nQuery History:")
            for i, (query, agent_choice, correct_choice) in enumerate(zip(
                query_history[:len(correct_choices)], 
                agent_choices[:len(correct_choices)], 
                correct_choices), 1):
                
                status = "CORRECT" if agent_choice == correct_choice else "WRONG"
                print(f"   {i}. {status} '{query[:50]}' -> Agent: {agent_choice}, Correct: {correct_choice}")
    
    def display(self):
        # Create layout
        url_section = widgets.VBox([
            widgets.HTML("<h3>Step 1: Load Web Sources</h3>"),
            self.url_input,
            self.load_button,
            widgets.HTML("<h4>Progress:</h4>"),
            self.progress_output
        ])
        
        query_section = widgets.VBox([
            widgets.HTML("<h3>Step 2: Submit Query</h3>"),
            self.query_input,
            self.query_button,
            widgets.HTML("<h4>Response:</h4>"),
            self.response_output
        ])
        
        feedback_section = widgets.VBox([
            widgets.HTML("<h3>Step 3: Provide Feedback</h3>"),
            self.agent_was_correct,
            self.feedback_dropdown,
            self.feedback_button
        ])
        
        evaluation_section = widgets.VBox([
            widgets.HTML("<h3>Step 4: View Evaluation</h3>"),
            self.eval_button,
            self.evaluation_output
        ])
        
        main_layout = widgets.VBox([
            widgets.HTML("<h1>Agentic RAG System</h1>"),
            url_section,
            query_section,
            feedback_section,
            evaluation_section
        ], layout=widgets.Layout(max_width='90%'))
        
        display(main_layout)

In [132]:
def main():
    interface = UserInterface()
    interface.display()

In [133]:
if __name__ == "__main__":
    main()